# Breast Cancer Wisconsin — Model Training & Evaluation

This notebook trains **5 classification models** on the Breast Cancer Wisconsin (Diagnostic) dataset, evaluates them on **6 metrics**, and exports the held-out test split as `test_data.csv` for the Streamlit app.

**No serialized model files (.pkl) are produced** — the Streamlit app retrains the models in-memory at runtime.

**Models:** Logistic Regression, Decision Tree, kNN, Naive Bayes, Random Forest  
**Metrics:** Accuracy, AUC, Precision, Recall, F1, MCC

Each model is a `Pipeline(StandardScaler -> classifier)` so the raw feature CSV can be scored by any model without leakage (scaler is fit on train only).

## 1. Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report,
)

RANDOM_STATE = 42
TEST_SIZE = 0.20

## 2. Load the dataset

569 instances, 30 numeric features, binary target (0 = malignant, 1 = benign). The data ships inside scikit-learn, so there is no download.

In [2]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

print('Instances :', X.shape[0])
print('Features  :', X.shape[1])
print('Classes   :', dict(zip(data.target_names, [0, 1])))
print('Balance   :', dict(y.value_counts().sort_index()))
X.head()

Instances : 569
Features  : 30
Classes   : {'malignant': 0, 'benign': 1}
Balance   : {0: 212, 1: 357}


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## 3. Train / test split (stratified, 80/20)

A single shared split is used so every model is compared on identical data. The 20% test set is exported as `test_data.csv` for the app.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'model' else Path.cwd()

test_df = X_test.copy()
test_df['target'] = y_test.values
test_df.to_csv(PROJECT_ROOT / 'test_data.csv', index=False)
print('Train:', X_train.shape, '| Test:', X_test.shape)

Train: (455, 30) | Test: (114, 30)


## 4. Define the models

Every classifier is wrapped with `StandardScaler`. Scaling is essential for Logistic Regression and kNN, and harmless (monotonic) for the tree and Naive Bayes models.

In [4]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
    ]),
    'Decision Tree': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    'kNN': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=5)),
    ]),
    'Naive Bayes': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GaussianNB()),
    ]),
    'Random Forest (Ensemble)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)),
    ]),
}
list(models.keys())

['Logistic Regression',
 'Decision Tree',
 'kNN',
 'Naive Bayes',
 'Random Forest (Ensemble)']

## 5. Helper functions

In [5]:
def positive_class_scores(model, X):
    """Positive-class probability for AUC (with safe fallbacks)."""
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, 'decision_function'):
        return model.decision_function(X)
    return model.predict(X)

def compute_metrics(y_true, y_pred, y_score):
    return {
        'Accuracy': round(float(accuracy_score(y_true, y_pred)), 4),
        'AUC': round(float(roc_auc_score(y_true, y_score)), 4),
        'Precision': round(float(precision_score(y_true, y_pred)), 4),
        'Recall': round(float(recall_score(y_true, y_pred)), 4),
        'F1': round(float(f1_score(y_true, y_pred)), 4),
        'MCC': round(float(matthews_corrcoef(y_true, y_pred)), 4),
    }

## 6. Train & evaluate each model

Models are trained and scored, but **not** saved to disk.

In [6]:
all_metrics = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_score = positive_class_scores(model, X_test)
    all_metrics[name] = compute_metrics(y_test, y_pred, y_score)
    print(f'{name}: {all_metrics[name]}')

Logistic Regression: {'Accuracy': 0.9825, 'AUC': 0.9954, 'Precision': 0.9861, 'Recall': 0.9861, 'F1': 0.9861, 'MCC': 0.9623}
Decision Tree: {'Accuracy': 0.9123, 'AUC': 0.9157, 'Precision': 0.9559, 'Recall': 0.9028, 'F1': 0.9286, 'MCC': 0.8174}
kNN: {'Accuracy': 0.9561, 'AUC': 0.9788, 'Precision': 0.9589, 'Recall': 0.9722, 'F1': 0.9655, 'MCC': 0.9054}
Naive Bayes: {'Accuracy': 0.9298, 'AUC': 0.9868, 'Precision': 0.9444, 'Recall': 0.9444, 'F1': 0.9444, 'MCC': 0.8492}
Random Forest (Ensemble): {'Accuracy': 0.9561, 'AUC': 0.9932, 'Precision': 0.9589, 'Recall': 0.9722, 'F1': 0.9655, 'MCC': 0.9054}


## 7. Comparison table

In [7]:
results = pd.DataFrame(all_metrics).T.round(4)
results

,Accuracy,AUC,Precision,Recall,F1,MCC
Logistic Regression,0.9825,0.9954,0.9861,0.9861,0.9861,0.9623
Decision Tree,0.9123,0.9157,0.9559,0.9028,0.9286,0.8174
kNN,0.9561,0.9788,0.9589,0.9722,0.9655,0.9054
Naive Bayes,0.9298,0.9868,0.9444,0.9444,0.9444,0.8492
Random Forest (Ensemble),0.9561,0.9932,0.9589,0.9722,0.9655,0.9054


## 8. Confusion matrix & classification report (per model)

Example below for Random Forest; change the key to inspect any model.

In [8]:
name = 'Random Forest (Ensemble)'
model = models[name]
y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm,
    index=['Actual malignant (0)', 'Actual benign (1)'],
    columns=['Predicted malignant (0)', 'Predicted benign (1)'])
print('Confusion matrix -', name)
print(cm_df, '\n')
print(classification_report(y_test, y_pred,
      target_names=['malignant (0)', 'benign (1)']))

Confusion matrix - Random Forest (Ensemble)
                      Predicted malignant (0)  Predicted benign (1)
Actual malignant (0)                       39                     3
Actual benign (1)                           2                    70 

               precision    recall  f1-score   support

malignant (0)       0.95      0.93      0.94        42
   benign (1)       0.96      0.97      0.97        72

     accuracy                           0.96       114
    macro avg       0.96      0.95      0.95       114
 weighted avg       0.96      0.96      0.96       114



## 9. Observations

- **Logistic Regression** — best overall; classes are near-linearly separable after scaling.
- **Decision Tree** — weakest; a single unpruned tree overfits.
- **kNN** — strong and balanced; benefits from feature scaling.
- **Naive Bayes** — high AUC but lower accuracy (correlated features violate its independence assumption).
- **Random Forest** — very robust; ensembling fixes the single tree's variance.

**Overall winner: Logistic Regression** (top on all six metrics), with Random Forest a close, more robust runner-up.